# Session 06 · Built-in Functions in T-SQL

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - string functions

In [2]:
from campus_db import connect, show
con = connect()
TODAY = "2026-10-27"
show(con, """SELECT name,
       UPPER(name)                               AS upper_name,
       LENGTH(name)                              AS len,
       SUBSTR(name, 1, INSTR(name, ' ') - 1)     AS first_name,
       SUBSTR(name, INSTR(name, ' ') + 1)        AS last_name,
       REPLACE(COALESCE(email, ''), '@example.com', '@campus.pk') AS new_email,
       TRIM('  ' || city || '  ')                AS city_trimmed
FROM students LIMIT 5""")

name          | upper_name    | len | first_name | last_name | new_email        | city_trimmed
--------------+---------------+-----+------------+-----------+------------------+-------------
Ayesha Khan   | AYESHA KHAN   | 11  | Ayesha     | Khan      | ayesha@campus.pk | Karachi     
Bilal Ahmed   | BILAL AHMED   | 11  | Bilal      | Ahmed     |                  | Lahore      
Hira Siddiqui | HIRA SIDDIQUI | 13  | Hira       | Siddiqui  | hira@campus.pk   | Karachi     
Omar Farooq   | OMAR FAROOQ   | 11  | Omar       | Farooq    | omar@campus.pk   | Quetta      
Zara Hussain  | ZARA HUSSAIN  | 12  | Zara       | Hussain   |                  | Karachi     
(5 rows)



[('Ayesha Khan',
  'AYESHA KHAN',
  11,
  'Ayesha',
  'Khan',
  'ayesha@campus.pk',
  'Karachi'),
 ('Bilal Ahmed', 'BILAL AHMED', 11, 'Bilal', 'Ahmed', '', 'Lahore'),
 ('Hira Siddiqui',
  'HIRA SIDDIQUI',
  13,
  'Hira',
  'Siddiqui',
  'hira@campus.pk',
  'Karachi'),
 ('Omar Farooq',
  'OMAR FAROOQ',
  11,
  'Omar',
  'Farooq',
  'omar@campus.pk',
  'Quetta'),
 ('Zara Hussain', 'ZARA HUSSAIN', 12, 'Zara', 'Hussain', '', 'Karachi')]

## Example 2 - date functions

In [3]:
show(con, """SELECT name, joined_on,
       CAST(julianday(?) - julianday(joined_on) AS INTEGER)  AS days_enrolled,
       strftime('%Y', joined_on)                             AS join_year,
       strftime('%m', joined_on)                             AS join_month,
       date(joined_on, '+6 months')                          AS review_date,
       date(joined_on, 'start of month', '+1 month', '-1 day') AS month_end
FROM students ORDER BY joined_on LIMIT 5""", (TODAY,))

name         | joined_on  | days_enrolled | join_year | join_month | review_date | month_end 
-------------+------------+---------------+-----------+------------+-------------+-----------
Ayesha Khan  | 2026-01-05 | 295           | 2026      | 01         | 2026-07-05  | 2026-01-31
Bilal Ahmed  | 2026-01-07 | 293           | 2026      | 01         | 2026-07-07  | 2026-01-31
Ali Raza     | 2026-01-10 | 290           | 2026      | 01         | 2026-07-10  | 2026-01-31
Omar Farooq  | 2026-01-25 | 275           | 2026      | 01         | 2026-07-25  | 2026-01-31
Zara Hussain | 2026-01-30 | 270           | 2026      | 01         | 2026-07-30  | 2026-01-31
(5 rows)



[('Ayesha Khan', '2026-01-05', 295, '2026', '01', '2026-07-05', '2026-01-31'),
 ('Bilal Ahmed', '2026-01-07', 293, '2026', '01', '2026-07-07', '2026-01-31'),
 ('Ali Raza', '2026-01-10', 290, '2026', '01', '2026-07-10', '2026-01-31'),
 ('Omar Farooq', '2026-01-25', 275, '2026', '01', '2026-07-25', '2026-01-31'),
 ('Zara Hussain', '2026-01-30', 270, '2026', '01', '2026-07-30', '2026-01-31')]

## Example 3 - mathematical functions

In [4]:
show(con, """SELECT name, salary,
       ROUND(salary / 12.0, 2)        AS monthly,
       ROUND(salary * 0.075)          AS tax_estimate,
       CAST(salary / 1000 AS INTEGER) AS thousands_floor,
       ABS(salary - 180000)           AS distance_from_180k,
       salary % 7                     AS mod7
FROM faculty""")
show(con, "SELECT ROUND(AVG(marks), 2) AS mean, MAX(marks) - MIN(marks) AS range_, SQRT(AVG(marks*marks) - AVG(marks)*AVG(marks)) AS std_dev FROM marks")

name         | salary | monthly  | tax_estimate | thousands_floor | distance_from_180k | mod7
-------------+--------+----------+--------------+-----------------+--------------------+-----
Sana Iqbal   | 180000 | 15000.0  | 13500.0      | 180             | 0                  | 2   
Usman Tariq  | 165000 | 13750.0  | 12375.0      | 165             | 15000              | 3   
Mehwish Raza | 172000 | 14333.33 | 12900.0      | 172             | 8000               | 3   
Kamran Ali   | 150000 | 12500.0  | 11250.0      | 150             | 30000              | 4   
Nida Farooq  | 210000 | 17500.0  | 15750.0      | 210             | 30000              | 0   
(5 rows)

mean  | range_ | std_dev          
------+--------+------------------
72.06 | 56     | 16.60748607556243
(1 rows)



[(72.06, 56, 16.60748607556243)]

## Example 4 - aggregate and conversion functions together

In [5]:
show(con, """SELECT b.course,
       COUNT(s.student_id)                       AS students,
       SUM(s.fee_paid)                           AS collected,
       'Rs ' || format('%,d', SUM(s.fee_paid))   AS collected_text,
       group_concat(s.name, ', ')                AS names      -- T-SQL: STRING_AGG(s.name, ', ')
FROM batches b LEFT JOIN students s ON s.batch_id = b.batch_id
GROUP BY b.course""")

course           | students | collected | collected_text | names                                                         
-----------------+----------+-----------+----------------+---------------------------------------------------------------
Data Science     | 2        | 60000     | Rs 60,000      | Omar Farooq, Zara Hussain                                     
Diploma in AI    | 5        | 195000    | Rs 195,000     | Ali Raza, Ayesha Khan, Bilal Ahmed, Fatima Noor, Hira Siddiqui
Machine Learning | 0        | None      | Rs 0           | None                                                          
Web Development  | 2        | 15000     | Rs 15,000      | Hamza Malik, Mariam Javed                                     
(4 rows)



[('Data Science', 2, 60000, 'Rs 60,000', 'Omar Farooq, Zara Hussain'),
 ('Diploma in AI',
  5,
  195000,
  'Rs 195,000',
  'Ali Raza, Ayesha Khan, Bilal Ahmed, Fatima Noor, Hira Siddiqui'),
 ('Machine Learning', 0, None, 'Rs 0', None),
 ('Web Development', 2, 15000, 'Rs 15,000', 'Hamza Malik, Mariam Javed')]

## Example 5 - ranking (window) functions

In [6]:
show(con, """SELECT subject, student_id, marks,
       ROW_NUMBER() OVER (PARTITION BY subject ORDER BY marks DESC) AS row_no,
       RANK()       OVER (PARTITION BY subject ORDER BY marks DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY marks DESC)                      AS overall_dense,
       NTILE(4)     OVER (ORDER BY marks DESC)                      AS quartile
FROM marks ORDER BY subject, row_no""")

subject    | student_id | marks | row_no | rnk | overall_dense | quartile
-----------+------------+-------+--------+-----+---------------+---------
Python     | 103        | 92    | 1      | 1   | 2             | 1       
Python     | 101        | 88    | 2      | 2   | 4             | 1       
Python     | 105        | 77    | 3      | 3   | 8             | 2       
Python     | 107        | 74    | 4      | 4   | 9             | 3       
Python     | 102        | 64    | 5      | 5   | 12            | 3       
Python     | 104        | 49    | 6      | 6   | 14            | 4       
Python     | 108        | 39    | 7      | 7   | 16            | 4       
SQL        | 103        | 95    | 1      | 1   | 1             | 1       
SQL        | 107        | 81    | 2      | 2   | 6             | 2       
SQL        | 101        | 79    | 3      | 3   | 7             | 2       
SQL        | 102        | 58    | 4      | 4   | 13            | 4       
SQL        | 108        | 45    | 5   

[('Python', 103, 92, 1, 1, 2, 1),
 ('Python', 101, 88, 2, 2, 4, 1),
 ('Python', 105, 77, 3, 3, 8, 2),
 ('Python', 107, 74, 4, 4, 9, 3),
 ('Python', 102, 64, 5, 5, 12, 3),
 ('Python', 104, 49, 6, 6, 14, 4),
 ('Python', 108, 39, 7, 7, 16, 4),
 ('SQL', 103, 95, 1, 1, 1, 1),
 ('SQL', 107, 81, 2, 2, 6, 2),
 ('SQL', 101, 79, 3, 3, 7, 2),
 ('SQL', 102, 58, 4, 4, 13, 4),
 ('SQL', 108, 45, 5, 5, 15, 4),
 ('Statistics', 105, 83, 1, 1, 5, 2),
 ('Statistics', 104, 71, 2, 2, 10, 3),
 ('Web', 109, 90, 1, 1, 3, 1),
 ('Web', 106, 68, 2, 2, 11, 3)]

## Example 6 - aggregate window functions: running totals and share of total

In [7]:
show(con, """SELECT name, joined_on, fee_paid,
       SUM(fee_paid) OVER (ORDER BY joined_on)                         AS running_total,
       ROUND(100.0 * fee_paid / SUM(fee_paid) OVER (), 1)              AS pct_of_all,
       fee_paid - AVG(fee_paid) OVER (PARTITION BY batch_id)           AS vs_batch_avg,
       LAG(name) OVER (ORDER BY joined_on)                             AS joined_after
FROM students ORDER BY joined_on""")

name          | joined_on  | fee_paid | running_total | pct_of_all | vs_batch_avg | joined_after 
--------------+------------+----------+---------------+------------+--------------+--------------
Ayesha Khan   | 2026-01-05 | 45000    | 45000         | 16.7       | 10000.0      | None         
Bilal Ahmed   | 2026-01-07 | 30000    | 75000         | 11.1       | -5000.0      | Ayesha Khan  
Ali Raza      | 2026-01-10 | 30000    | 105000        | 11.1       | -5000.0      | Bilal Ahmed  
Omar Farooq   | 2026-01-25 | 15000    | 120000        | 5.6        | -15000.0     | Ali Raza     
Zara Hussain  | 2026-01-30 | 45000    | 165000        | 16.7       | 15000.0      | Omar Farooq  
Hamza Malik   | 2026-03-01 | 0        | 165000        | 0.0        | -7500.0      | Zara Hussain 
Mariam Javed  | 2026-03-05 | 15000    | 180000        | 5.6        | 7500.0       | Hamza Malik  
Hira Siddiqui | 2026-03-28 | 45000    | 225000        | 16.7       | 0.0          | Mariam Javed 
Fatima Noor   | 2026

[('Ayesha Khan', '2026-01-05', 45000, 45000, 16.7, 10000.0, None),
 ('Bilal Ahmed', '2026-01-07', 30000, 75000, 11.1, -5000.0, 'Ayesha Khan'),
 ('Ali Raza', '2026-01-10', 30000, 105000, 11.1, -5000.0, 'Bilal Ahmed'),
 ('Omar Farooq', '2026-01-25', 15000, 120000, 5.6, -15000.0, 'Ali Raza'),
 ('Zara Hussain', '2026-01-30', 45000, 165000, 16.7, 15000.0, 'Omar Farooq'),
 ('Hamza Malik', '2026-03-01', 0, 165000, 0.0, -7500.0, 'Zara Hussain'),
 ('Mariam Javed', '2026-03-05', 15000, 180000, 5.6, 7500.0, 'Hamza Malik'),
 ('Hira Siddiqui', '2026-03-28', 45000, 225000, 16.7, 0.0, 'Mariam Javed'),
 ('Fatima Noor', '2026-04-01', 45000, 270000, 16.7, 0.0, 'Hira Siddiqui'),
 ('Saad Qureshi', '2026-10-20', 0, 270000, 0.0, 0.0, 'Fatima Noor')]